In [1]:
import os, re, pickle
from collections import Counter
import numpy as np
import faiss
import pymupdf
from sentence_transformers import SentenceTransformer

DATA = "../data"
PAPERS = f"{DATA}/papers"
TARGET, MAX_CHARS, OVERLAP_SENT = 800, 1000, 1
CUT_REFS = True

old = pickle.load(open(f"{DATA}/all_chunks.pkl", "rb"))
old_n = Counter(c["source"] for c in old)
print("Old corpus:", len(old), "chunks")

/home/rgukt/Desktop/research-assistant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Old corpus: 1356 chunks


In [2]:
def cut_references(raw):
    m = None
    for m in re.finditer(r"\n\s*(References|REFERENCES|Bibliography)\s*\n", raw):
        pass
    if m and m.start() > 0.5 * len(raw):
        return raw[:m.start()]
    return raw

def split_sentences(t):
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9\[(“"])', t)
    out = []
    for s in parts:
        s = s.strip()
        out += [s[i:i + MAX_CHARS] for i in range(0, len(s), MAX_CHARS)]  # split junk "sentences"
    return [s for s in out if s]

def chunk_sentences(text, target=TARGET, overlap=OVERLAP_SENT):
    chunks, cur, n = [], [], 0
    for s in split_sentences(text):
        if cur and n + len(s) > target:
            chunks.append(" ".join(cur))
            cur = cur[-overlap:] if overlap else []
            if cur and sum(len(x) for x in cur) + len(s) > MAX_CHARS:
                cur = []
            n = sum(len(x) + 1 for x in cur)
        cur.append(s)
        n += len(s) + 1
    if cur:
        chunks.append(" ".join(cur))
    return [c for c in chunks if len(c) >= 100]

print("functions ready")

functions ready


In [3]:
all_chunks_v2 = []
for fname in sorted(os.listdir(PAPERS)):
    if not fname.endswith(".pdf"):
        continue
    doc = pymupdf.open(os.path.join(PAPERS, fname))
    raw = "".join(p.get_text() for p in doc)
    if CUT_REFS:
        raw = cut_references(raw)
    text = re.sub(r"\s+", " ", re.sub(r"-\n(?=[a-z])", "", raw))   # fix hyphenation, collapse whitespace
    pieces = chunk_sentences(text)
    all_chunks_v2 += [{"text": c, "source": fname} for c in pieces]
    avg = sum(map(len, pieces)) // max(1, len(pieces))
    print(f"{fname}: {old_n[fname]} -> {len(pieces)} chunks (avg {avg} chars)")

print("\nNew corpus:", len(all_chunks_v2), "chunks")

1811.08772v1.pdf: 145 -> 91 chunks (avg 723 chars)
1912.02145v1.pdf: 80 -> 42 chunks (avg 731 chars)
2108.12898v1.pdf: 73 -> 39 chunks (avg 665 chars)
2404.14464v1.pdf: 143 -> 65 chunks (avg 715 chars)
2504.13684v1.pdf: 67 -> 38 chunks (avg 685 chars)
2504.17204v1.pdf: 46 -> 28 chunks (avg 694 chars)
2506.06962v3.pdf: 172 -> 70 chunks (avg 707 chars)
2507.23334v2.pdf: 93 -> 64 chunks (avg 713 chars)
2510.25621v1.pdf: 325 -> 131 chunks (avg 710 chars)
2601.07528v2.pdf: 212 -> 158 chunks (avg 707 chars)

New corpus: 726 chunks


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c["text"] for c in all_chunks_v2]
emb = model.encode(texts, normalize_embeddings=True, batch_size=32, show_progress_bar=True)

index_v2 = faiss.IndexFlatIP(emb.shape[1])
index_v2.add(np.array(emb, dtype="float32"))

with open(f"{DATA}/all_chunks_v2.pkl", "wb") as f:
    pickle.dump(all_chunks_v2, f)
faiss.write_index(index_v2, f"{DATA}/all_index_v2.faiss")
print("Saved:", len(all_chunks_v2), "chunks |", index_v2.ntotal, "vectors")

Batches: 100%|█████████████████████████████████| 23/23 [01:35<00:00,  4.16s/it]

Saved: 726 chunks | 726 vectors


In [5]:
hits = [c for c in all_chunks_v2 if "faiss" in c["text"].lower()]
print(len(hits), "chunk(s) mention FAISS")
for h in hits:
    print(h["source"], "\n", h["text"], "\n")

1 chunk(s) mention FAISS
2506.06962v3.pdf 
 Each image is encoded into a sequence of patch-level representations and image tokens using the same image tokenizer employed in the Janus-Pro model. For efficient similarity search, we implement our retriever using the FAISS library [21]. Training Setup We adopt Janus-Pro-1B [9] and Show-o [44] as our backbone models and fine-tune them on a dataset of 50,000 image-caption pairs sampled from CC12M [5] and Midjourney-v6 3. We empirically determine the optimal hyperparameters for DAiD and FAiD, and the complete hyperparameter optimization experiment results can be found in Appendix C.2. Further details regarding the training dataset construction and implementation can be found in Appendix B.3. 

